# Niveau 1 - Tâche 1 : Prétraitement de données pour le Machine Learning

**Dataset choisi : Churn Prediction Data (Telco, format bigml)**

Ce dataset est idéal pour cette tâche car il contient :
- des variables catégorielles (`State`, `International plan`, `Voice mail plan`)
- des variables numériques à échelles très différentes (minutes, appels, charges)
- une cible binaire (`Churn`)

**Objectifs couverts :**
1. Gestion des valeurs manquantes
2. Encodage des variables catégorielles
3. Normalisation / standardisation des variables numériques
4. Split train/test

**Outils :** Python, pandas, scikit-learn


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

pd.set_option('display.max_columns', None)

df = pd.read_csv('data/churn_80.csv')
print("Shape:", df.shape)
df.head()


Shape: (2666, 20)


,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


## 1. Exploration rapide et valeurs manquantes

Le dataset original ne contient aucune valeur manquante. Pour illustrer une pipeline de prétraitement réaliste (le jeu de données réel en contiendra souvent), nous injectons volontairement quelques valeurs manquantes dans des colonnes numériques et catégorielles, puis nous les traitons.

In [2]:
print("Valeurs manquantes avant simulation :")
print(df.isna().sum().sum())

# Simulation de valeurs manquantes (à but pédagogique)
rng = np.random.RandomState(42)
df_sim = df.copy()
for col in ['Total day minutes', 'Total eve calls', 'Voice mail plan']:
    idx = rng.choice(df_sim.index, size=int(0.03 * len(df_sim)), replace=False)
    df_sim.loc[idx, col] = np.nan

print("\nValeurs manquantes après simulation :")
print(df_sim.isna().sum()[df_sim.isna().sum() > 0])


Valeurs manquantes avant simulation :
0

Valeurs manquantes après simulation :
Voice mail plan      79
Total day minutes    79
Total eve calls      79
dtype: int64


In [3]:
# Gestion des valeurs manquantes
# - colonnes numériques -> imputation par la médiane
# - colonnes catégorielles -> imputation par le mode

num_cols = df_sim.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_sim.select_dtypes(include=['object']).columns.tolist()

for c in num_cols:
    if df_sim[c].isna().sum() > 0:
        df_sim[c] = df_sim[c].fillna(df_sim[c].median())

for c in cat_cols:
    if df_sim[c].isna().sum() > 0:
        df_sim[c] = df_sim[c].fillna(df_sim[c].mode()[0])

print("Valeurs manquantes restantes :", df_sim.isna().sum().sum())


Valeurs manquantes restantes : 0


/tmp/ipykernel_636/1599253668.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_sim.select_dtypes(include=['object']).columns.tolist()


## 2. Encodage des variables catégorielles

- `International plan` et `Voice mail plan` : binaires -> Label Encoding (Yes/No)
- `State` : nominale à haute cardinalité -> One-Hot Encoding
- `Churn` (cible) : Label Encoding (True/False -> 1/0)

In [4]:
df_enc = df_sim.copy()

le = LabelEncoder()
for c in ['International plan', 'Voice mail plan']:
    df_enc[c] = le.fit_transform(df_enc[c])

df_enc['Churn'] = df_enc['Churn'].astype(int)

df_enc = pd.get_dummies(df_enc, columns=['State'], prefix='state', drop_first=True)

print("Shape après encodage :", df_enc.shape)
df_enc.head()


Shape après encodage : (2666, 69)


,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn,state_AL,state_AR,state_AZ,state_CA,state_CO,state_CT,state_DC,state_DE,state_FL,state_GA,state_HI,state_IA,state_ID,state_IL,state_IN,state_KS,state_KY,state_LA,state_MA,state_MD,state_ME,state_MI,state_MN,state_MO,state_MS,state_MT,state_NC,state_ND,state_NE,state_NH,state_NJ,state_NM,state_NV,state_NY,state_OH,state_OK,state_OR,state_PA,state_RI,state_SC,state_SD,state_TN,state_TX,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY
0,128,415,0,1,25,265.1,110,45.07,197.4,99.0,16.78,244.7,91,11.01,10.0,3,2.70,1,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,107,415,0,1,26,161.6,123,27.47,195.5,103.0,16.62,254.4,103,11.45,13.7,3,3.70,1,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,137,415,0,0,0,243.4,114,41.38,121.2,110.0,10.30,162.6,104,7.32,12.2,5,3.29,0,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,84,408,1,0,0,299.4,71,50.90,61.9,88.0,5.26,196.9,89,8.86,6.6,7,1.78,2,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,75,415,1,0,0,166.7,113,28.34,148.3,122.0,12.61,186.9,121,8.41,10.1,3,2.73,3,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False


## 3. Normalisation des variables numériques

In [5]:
target = 'Churn'
feature_cols = [c for c in df_enc.columns if c != target]

X = df_enc[feature_cols]
y = df_enc[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = [c for c in num_cols if c in X_train.columns]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

print("Train shape:", X_train_scaled.shape, " Test shape:", X_test_scaled.shape)
X_train_scaled[numeric_features].describe().T[['mean', 'std']].head()


Train shape: (2132, 68)  Test shape: (534, 68)


,mean,std
Account length,7.332054e-17,1.000235
Area code,1.533066e-16,1.000235
Number vmail messages,-3.999302e-17,1.000235
Total day minutes,-1.299773e-16,1.000235
Total day calls,3.341084e-16,1.000235


## 4. Split train/test — Résumé final

In [6]:
print(f"Train set : {X_train_scaled.shape[0]} lignes, {X_train_scaled.shape[1]} features")
print(f"Test set  : {X_test_scaled.shape[0]} lignes")
print(f"Distribution de la cible (train) :\n{y_train.value_counts(normalize=True)}")
print(f"\nDistribution de la cible (test) :\n{y_test.value_counts(normalize=True)}")

# Sauvegarde des jeux prétraités pour réutilisation éventuelle
X_train_scaled.to_csv('churn_train_preprocessed.csv', index=False)
X_test_scaled.to_csv('churn_test_preprocessed.csv', index=False)
print("\nFichiers prétraités sauvegardés.")


Train set : 2132 lignes, 68 features
Test set  : 534 lignes
Distribution de la cible (train) :
Churn
0    0.854597
1    0.145403
Name: proportion, dtype: float64

Distribution de la cible (test) :
Churn
0    0.853933
1    0.146067
Name: proportion, dtype: float64



Fichiers prétraités sauvegardés.


## Conclusion

Le pipeline de prétraitement complet a été appliqué :
- Imputation des valeurs manquantes (médiane / mode)
- Encodage des variables catégorielles (Label Encoding + One-Hot Encoding)
- Standardisation des variables numériques (StandardScaler)
- Split stratifié train/test (80/20)

Le dataset est maintenant prêt pour l'entraînement de modèles de classification (voir notebooks suivants : régression logistique, Random Forest, SVM).
